[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/masthead-data/templates/blob/main/notebooks/save_on_unused_storage.ipynb)

In [ ]:
# @title Get storage usage by dead end and unused tables

from google.cloud import bigquery
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = "" # @param {"type":"string","placeholder":"Enter the ID of your BigQuery project"}
DATASET_NAME = "" # @param {"type":"string","placeholder":"Enter the name of source dataset in Masthead Data project"}

bq_client = bigquery.Client(project=PROJECT_ID)
df_tables = bq_client.query_and_wait(f"""
SELECT
  subtype,
  project_id,
  target_resource,
  SAFE.STRING(operations[0].resource_type) AS resource_type,
  SAFE.INT64(overview.num_bytes) / POW(1024, 4) AS total_tib,
  SAFE.FLOAT64(overview.cost_30d) AS cost_usd_30d,
  SAFE.FLOAT64(overview.savings_30d) AS savings_usd_30d,
  'review pending' AS status
FROM `masthead-prod`.{DATASET_NAME}.insights
WHERE category = 'Cost'
  AND subtype IN ('Dead end table', 'Unused table')
  AND overview.num_bytes IS NOT NULL
ORDER BY total_tib DESC;
""").to_dataframe()

tables_to_review = df_tables.loc[df_tables['savings_usd_30d'] > 1]
tables_to_review.head(5)

In [ ]:
# @title Export tables into a Google Spreadsheet to review
# Requires Google Drive API and Google Sheets API enabled in the GCP project and credentials with access to those APIs

from google.auth import default

import gspread
import pandas as pd

credentials, _ = default()
gs_client = gspread.authorize(credentials)

spreadsheet = gs_client.create('Masthead Data: Storage Cost Optimization')
worksheet = spreadsheet.sheet1
worksheet.update_title(title="Unused Tables to Review")
worksheet.update([tables_to_review.columns.values.tolist()] + tables_to_review.values.tolist())

print(f"Worksheet URL: {worksheet.url}")

In [ ]:
# @title Pull back reviewed data from spreadsheet and drop tables

worksheet = gs_client.open('Masthead Data: Storage Cost Optimization').sheet1
df_from_sheets = pd.DataFrame(worksheet.get_all_records())

df_tables_to_drop = df_from_sheets.loc[df_from_sheets['status'] == 'to drop']

for _, row in df_tables_to_drop.iterrows():
    table_name = row['target_resource']
    resource_type = row['resource_type']
    ddl = f"DROP {resource_type} IF EXISTS `{table_name}`;"
    print(ddl)

    # DANGER ZONE: Uncomment the line below to execute the DROP statements
    # bq_client.query(ddl).result()